# 14 · MLflow - Week 2: Full Pipeline Tracking & Model Registry

Track every model of the platform in one experiment, store the churn model in the MLflow **Model Registry**, and pick the champion programmatically.

In [1]:
import sys, warnings, logging
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 30)

from src.config import *
from src.ingestion.etl import load_processed
D = load_processed()
tx, customers, products, inventory = D["transactions"], D["customers"], D["products"], D["inventory"]
daily = D["daily"]
print(f"Loaded {len(tx):,} clean transaction lines | {tx.customer_id.nunique():,} customers | "
      f"{tx.order_date.min().date()} -> {tx.order_date.max().date()}")

Loaded 54,339 clean transaction lines | 1,455 customers | 2024-01-01 -> 2025-12-31


In [2]:
series = daily.set_index("date")["units"].astype(float)
train, test = series.iloc[:-HOLDOUT_DAYS], series.iloc[-HOLDOUT_DAYS:]

## 1. One experiment, every model

In [3]:
import mlflow, mlflow.sklearn
from src import tracking
from src.forecasting.ensemble import available_models, run_forecasting
from src.churn.model import train_churn_model
tracking.setup(True)
mlflow.set_experiment("retailpulse-week2")

res = run_forecasting(series, available_models())
for name, row in res["metrics"].iterrows():
    with mlflow.start_run(run_name=f"forecast_{name.split()[0]}"):
        mlflow.set_tags({"task": "demand_forecast", "stage": "candidate"})
        mlflow.log_params({"model": name, "holdout_days": HOLDOUT_DAYS, "weights": res["weights"].to_dict() if "ensemble" in name else "n/a"})
        mlflow.log_metrics(row.to_dict())
res["metrics"].round(3)

,mae,rmse,wape,bias
gbm,15.649,19.848,0.122,4.213
prophet,14.030,18.108,0.110,3.080
lstm,17.623,23.873,0.138,-12.855
ensemble (hybrid),13.099,17.140,0.102,-1.255


## 2. Churn model + registry

In [4]:
bundle = train_churn_model(tx, customers)
with mlflow.start_run(run_name="churn_gbm") as run:
    mlflow.set_tags({"task": "churn", "stage": "candidate"})
    mlflow.log_params(bundle["params"])
    mlflow.log_metrics(bundle["metrics"])
    mlflow.log_dict(bundle["importance"].round(4).to_dict(), "feature_importance.json")
    info = mlflow.sklearn.log_model(bundle["model"], name="churn_model", registered_model_name="retailpulse-churn",
                                serialization_format="cloudpickle")  # our own trusted model
print("logged model:", info.model_uri)

2026/10/17 12:38:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


logged model: models:/m-6073b789aeda4ce1a4508d341c7fde6a


Successfully registered model 'retailpulse-churn'.
Created version '1' of model 'retailpulse-churn'.


## 3. Compare runs and choose a champion

In [5]:
runs = mlflow.search_runs(experiment_names=["retailpulse-week2"])
fc_runs = runs[runs["tags.task"] == "demand_forecast"].sort_values("metrics.mae")
fc_runs[["tags.mlflow.runName", "metrics.mae", "metrics.rmse", "metrics.wape"]].round(3)

,tags.mlflow.runName,metrics.mae,metrics.rmse,metrics.wape
1,forecast_ensemble,13.099,17.140,0.102
6,forecast_ensemble,13.099,17.140,0.102
8,forecast_prophet,14.030,18.108,0.110
3,forecast_prophet,14.030,18.108,0.110
9,forecast_gbm,15.649,19.848,0.122
4,forecast_gbm,15.649,19.848,0.122
2,forecast_lstm,17.623,23.873,0.138
7,forecast_lstm,17.623,23.873,0.138


In [6]:
champion = fc_runs.iloc[0]
print("Champion forecaster:", champion["tags.mlflow.runName"], "| MAE", round(champion["metrics.mae"], 3))
client = mlflow.MlflowClient()
client.set_tag(champion["run_id"], "stage", "champion")

Champion forecaster: forecast_ensemble | MAE 13.099


## 4. Load the registered model back (what a serving job would do)

In [7]:
versions = client.search_model_versions("name='retailpulse-churn'")
latest = max(versions, key=lambda v: int(v.version))
loaded = mlflow.sklearn.load_model(f"models:/retailpulse-churn/{latest.version}")
from src.churn.model import make_training_set
Xc, yc, _ = make_training_set(tx, customers)
print("registry version:", latest.version, "| sample churn probabilities:", loaded.predict_proba(Xc.head(5))[:, 1].round(3))

registry version: 1 | sample churn probabilities: [0.965 0.075 0.009 0.029 0.014]


## 5. CI/CD & monitoring hooks
- CI (`.github/workflows/ci.yml`) runs tests and builds the Docker image on every push.
- A scheduled retrain job runs `scripts/run_pipeline.py --mlflow`; drift alerts (notebook 12) decide whether to promote a new champion.